# 04 — Modelos evaluados (§8 del README)

| | |
|---|---|
| **Entrada** | Split de §6 (`notebooks/02_preprocessing.ipynb`); baseline de §7 |
| **Objetivo** | Comparar familias de modelos bajo **un mismo split** y elegir los que pasan a validación cruzada (§9) |
| **Salida** | Ranking preliminar por ROC-AUC y PR-AUC para README §8 |

**Reglas:** sin resultados inventados. Esta es una **comparación exploratoria** en holdout; el ajuste de hiperparámetros y la validación cruzada son §9.

## 1. Selección de modelos y justificación

| Modelo | Por qué se propone | Preprocesamiento | Ventajas | Limitaciones |
|---|---|---|---|---|
| **Regresión logística** | Baseline lineal, probabilístico e interpretable | Escalado + One-Hot | Coeficientes legibles, rápido | Supone relaciones lineales |
| **Árbol de decisión** | Reglas no lineales simples | Solo One-Hot | Muy interpretable (`max_depth` limita sobreajuste) | Inestable, sesgo por clase |
| **Random Forest** | Ensamble de árboles, robusto | Solo One-Hot | Maneja no linealidad y atípicos; importancias | Caja seminegra, más costoso |
| **Gradient Boosting** | Ensamble secuencial de alto desempeño | Solo One-Hot | Suele lograr el mejor PR-AUC | Puede sobreajustar sin control |
| **KNN** | Referencia no paramétrica | **Escalado obligatorio** | Simple, captura vecindarios | Costoso en predicción; sensible a dimensionalidad |
| **Dummy/Baseline (§7)** | Referencia | — | — | — |

Todos se construyen con `src/models.py::build_pipeline`, que incluye el preprocesador de §6 **dentro** del pipeline (el escalado se ajusta solo donde corresponde).

**Hiperparámetros iniciales** (sin tuning, para comparación justa): logística `C=1, max_iter=1000`; árbol `max_depth=6`; Random Forest `n_estimators=400, min_samples_leaf=5, class_weight='balanced_subsample'`; Gradient Boosting por defecto; KNN `n_neighbors=25`.

In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from src.models import build_candidate_models
from src.preprocessing import (
    RANDOM_STATE,
    clean_data,
    load_raw_data,
    split_features_target,
    train_test_split_data,
)

df = clean_data(load_raw_data())
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = train_test_split_data(X, y)

print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"Features: {X.shape[1]} -> {len(__import__('src.preprocessing', fromlist=['build_preprocessor']).build_preprocessor().fit(X_train).get_feature_names_out())} columnas transformadas")

Train: 32,940 | Test: 8,236
Features: 13 -> 40 columnas transformadas


## 2. Comparación inicial en holdout

Se entrena cada modelo con el mismo `X_train` y se evalúa en el mismo `X_test`, con el umbral por defecto (0,5). Se mide también el **tiempo de entrenamiento** como referencia de costo computacional.

In [2]:
modelos = build_candidate_models()
modelos["Dummy (prior)"] = DummyClassifier(strategy="prior", random_state=RANDOM_STATE)

filas = []
preds = {}
for nombre, modelo in modelos.items():
    t0 = time.perf_counter()
    modelo.fit(X_train, y_train)
    t_train = time.perf_counter() - t0

    y_pred = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)[:, 1]
    preds[nombre] = (y_pred, y_proba)

    filas.append({
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
        "PR-AUC": average_precision_score(y_test, y_proba),
        "t_train_s": t_train,
    })

tabla = pd.DataFrame(filas).set_index("Modelo").sort_values("PR-AUC", ascending=False)
tabla.round(3)

,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC,t_train_s
Modelo,,,,,,,
Random Forest,0.865,0.432,0.628,0.512,0.794,0.453,7.859
Gradient Boosting,0.897,0.640,0.197,0.301,0.808,0.453,10.102
Regresion logistica,0.898,0.693,0.168,0.271,0.776,0.430,0.381
Regresion logistica balanceada,0.773,0.290,0.699,0.410,0.777,0.429,0.456
Arbol de decision,0.897,0.668,0.178,0.281,0.791,0.407,0.351
KNN,0.895,0.609,0.184,0.283,0.769,0.405,0.092
Dummy (prior),0.887,0.000,0.000,0.000,0.500,0.113,0.002


## 3. Lectura de resultados

Con desbalance 7,9:1, la comparación se ordena por **PR-AUC** (resume precision/recall de la clase minoritaria). El accuracy se muestra como control, no como criterio.

In [3]:
print("Ranking por PR-AUC (clase positiva = yes):")
print(tabla[["PR-AUC", "ROC-AUC", "Recall", "F1", "Accuracy"]].round(3).to_string())

base_pr = tabla.loc["Dummy (prior)", "PR-AUC"]
print(f"\nPR-AUC de la referencia trivial: {base_pr:.3f}")
mejores = tabla.drop(index="Dummy (prior)").sort_values("PR-AUC", ascending=False).head(3)
print("\nTop-3 modelos por PR-AUC:")
print(mejores[["PR-AUC", "ROC-AUC", "Recall", "F1"]].round(3).to_string())

Ranking por PR-AUC (clase positiva = yes):
                                PR-AUC  ROC-AUC  Recall     F1  Accuracy
Modelo                                                                  
Random Forest                    0.453    0.794   0.628  0.512     0.865
Gradient Boosting                0.453    0.808   0.197  0.301     0.897
Regresion logistica              0.430    0.776   0.168  0.271     0.898
Regresion logistica balanceada   0.429    0.777   0.699  0.410     0.773
Arbol de decision                0.407    0.791   0.178  0.281     0.897
KNN                              0.405    0.769   0.184  0.283     0.895
Dummy (prior)                    0.113    0.500   0.000  0.000     0.887

PR-AUC de la referencia trivial: 0.113

Top-3 modelos por PR-AUC:
                     PR-AUC  ROC-AUC  Recall     F1
Modelo                                             
Random Forest         0.453    0.794   0.628  0.512
Gradient Boosting     0.453    0.808   0.197  0.301
Regresion logistica   0.4

## 4. Interpretabilidad — qué aprende la regresión logística

Los coeficientes permiten leer la **dirección** de la asociación con `yes` (relación descriptiva, no causal). Numéricas estandarizadas y categóricas One-Hot.

In [4]:
log = modelos["Regresion logistica"]
nombres = log.named_steps["preprocessor"].get_feature_names_out()
coef = pd.Series(log.named_steps["model"].coef_[0], index=nombres)
top = coef.reindex(coef.abs().sort_values(ascending=False).index).head(12)
print("Coeficientes de mayor magnitud (|beta|):")
print(top.round(3).to_string())

Coeficientes de mayor magnitud (|beta|):
poutcome_failure       -1.077
emp.var.rate           -0.793
poutcome_success        0.779
poutcome_nonexistent   -0.518
nr.employed            -0.373
marital_divorced       -0.359
job_retired             0.306
cons.price.idx          0.303
job_blue-collar        -0.296
marital_married        -0.290
housing_yes            -0.287
housing_unknown        -0.276


## 5. Importancia de variables — Random Forest

Las importancias (impureza) dan una señal complementaria no lineal. Se agregan por variable original para comparar con los coeficientes.

In [5]:
rf = modelos["Random Forest"]
nombres_rf = rf.named_steps["preprocessor"].get_feature_names_out()
imp = pd.Series(rf.named_steps["model"].feature_importances_, index=nombres_rf)

grupo = imp.groupby(lambda c: c.split("_")[0]).sum().sort_values(ascending=False)
print("Importancia agregada por variable (suma de sus columnas One-Hot):")
print(grupo.round(4).to_string())

top15 = imp.sort_values(ascending=False).head(15)
print("\nTop-15 columnas transformadas por importancia:")
print(top15.round(4).to_string())

Importancia agregada por variable (suma de sus columnas One-Hot):
euribor3m         0.2077
nr.employed       0.1445
age               0.1325
emp.var.rate      0.0889
job               0.0793
cons.conf.idx     0.0687
poutcome          0.0639
education         0.0638
cons.price.idx    0.0501
marital           0.0327
housing           0.0300
loan              0.0219
previous          0.0162

Top-15 columnas transformadas por importancia:
euribor3m                      0.2077
nr.employed                    0.1445
age                            0.1325
emp.var.rate                   0.0889
cons.conf.idx                  0.0687
cons.price.idx                 0.0501
poutcome_success               0.0374
poutcome_nonexistent           0.0165
previous                       0.0162
housing_yes                    0.0142
housing_no                     0.0141
education_university.degree    0.0135
job_blue-collar                0.0131
job_admin.                     0.0131
marital_married              

## 6. Conclusión de §8 y candidatos para §9

Criterios para pasar a validación cruzada (§9):

- Buen **PR-AUC/ROC-AUC** en holdout.
- Si la mejor variante usa `class_weight='balanced'` (mejor recall), se evalúan ambas en §9 para decidir con evidencia.
- Se descartan los que no superan claramente a la referencia trivial.
- El costo computacional se considera (KNN y GB son más costosos en predicción/entrenamiento).

La lista final de candidatos queda en la última celda de este notebook.

In [6]:
candidatos = tabla.drop(index="Dummy (prior)").sort_values("PR-AUC", ascending=False).index.tolist()
print("Modelos que pasan a §9 (orden por PR-AUC):")
for m in candidatos:
    print(f"  - {m}")

Modelos que pasan a §9 (orden por PR-AUC):
  - Random Forest
  - Gradient Boosting
  - Regresion logistica
  - Regresion logistica balanceada
  - Arbol de decision
  - KNN
